# LC 23 — Merge K Sorted Lists

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A min-heap lets you always grab the
globally smallest node across all lists in O(log k) time, so you
never do a full scan — just pop, append, and push the next node
from the same list.
</div>

## Official Problem Statement

You are given an array of `k` linked-lists `lists`, each linked-list
is sorted in ascending order.

*Merge all the linked-lists into one sorted linked-list and return it.*

**Constraints:**
- `k == lists.length`
- `0 <= k <= 10^4`
- `0 <= lists[i].length <= 500`
- `-10^4 <= lists[i][j] <= 10^4`
- `lists[i]` is sorted in ascending order
- The sum of `lists[i].length` will not exceed `10^4`

## What This Is Actually Asking

You have k sorted streams and you need one merged sorted stream.
The naive approach — merge two at a time — works but wastes time
because you keep revisiting nodes. Instead, maintain a frontier of
exactly one candidate per list and always advance the smallest.
The heap is just an efficient "who is smallest right now" oracle
that updates in O(log k) instead of O(k) per step.

## Walk Through an Example by Hand

```
lists = [[1,4,5], [1,3,4], [2,6]]

Step 0 — push heads:  heap = [(1,0,node1), (1,1,node1), (2,2,node2)]

Step 1 — pop (1,0): append 1, push 4 from list-0
         heap = [(1,1,node1), (2,2,node2), (4,0,node4)]

Step 2 — pop (1,1): append 1, push 3 from list-1
         heap = [(2,2,node2), (3,1,node3), (4,0,node4)]

Step 3 — pop (2,2): append 2, push 6 from list-2
         heap = [(3,1,node3), (4,0,node4), (6,2,node6)]

... continue until heap empty

Result: 1 -> 1 -> 2 -> 3 -> 4 -> 4 -> 5 -> 6
```

## The Picture

```
List 0:  1 --> 4 --> 5
List 1:  1 --> 3 --> 4
List 2:  2 --> 6

Min-Heap (val, list_idx, node):
┌─────────────────────────┐
│   (1, 0, *)             │  <- pop this
│   (1, 1, *)             │
│   (2, 2, *)             │
└─────────────────────────┘
         ↓  pop + push next from list 0
┌─────────────────────────┐
│   (1, 1, *)             │  <- pop this next
│   (2, 2, *)             │
│   (4, 0, *)             │
└─────────────────────────┘

Dummy --> [1] --> [1] --> [2] --> [3] --> [4] --> [4] --> [5] --> [6]
          ↑ result built left to right
```

## When To Use This Pattern

- When you have **k sorted sequences** and need a single merged
  stream, think **min-heap / priority queue**.
- When you need the **global minimum across multiple fronts**
  efficiently, think **heap**.
- When naive O(kN) feels slow and k is large, think
  **O(N log k) with a heap**.
- When merging database query results or log streams from
  multiple sources, think **k-way merge**.
- When you see "k sorted lists/arrays/streams", think
  **this exact pattern**.

## The Approach

Push the first node of every non-empty list onto a min-heap as
(val, list_index, node). Repeatedly pop the smallest tuple,
append that node to the result, and push the popped node's
`.next` if it exists. The list_index breaks ties so Python never
tries to compare TreeNode objects.

In [ ]:
import heapq
from collections import deque
from typing import Optional, List


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def make_list(vals):
    dummy = ListNode(0)
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def to_list(head):
    r = []
    while head:
        r.append(head.val)
        head = head.next
    return r

In [ ]:
def test_harness(func):
    cases = [
        # (input_as_list_of_lists, expected_output)
        ([[1, 4, 5], [1, 3, 4], [2, 6]],
         [1, 1, 2, 3, 4, 4, 5, 6]),
        ([],
         []),
        ([[]],
         []),
        ([[1]],
         [1]),
        ([[-1, 0], [-2, -1, 0, 1]],
         [-2, -1, -1, 0, 0, 1]),
    ]
    passed = 0
    for i, (raw_lists, expected) in enumerate(cases):
        linked = [make_list(lst) for lst in raw_lists]
        result = to_list(func(linked))
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"  Case {i}: got {result}, expected {expected}")
        print(f"  Case {i}: {status}")
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
def merge_k_lists(
    lists: List[Optional[ListNode]]
) -> Optional[ListNode]:
    """
    Merge k sorted linked lists using a min-heap.

    Args:
        lists: list of heads of sorted linked lists

    Returns:
        head of merged sorted linked list

    Strategy:
        - Push (val, idx, node) for each list head onto heap
        - Pop min, append to result, push node.next if exists
        - idx breaks ties so we never compare ListNode objects
    """
    heap = []
    # Debug: show initial heap population
    print("[DEBUG] Initialising heap with heads:")
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
            print(f"  pushed ({node.val}, list={i})")

    dummy = ListNode(0)
    cur = dummy

    while heap:
        val, idx, node = heapq.heappop(heap)
        print(f"[DEBUG] popped val={val} from list {idx}")
        cur.next = node
        cur = cur.next
        if node.next:
            heapq.heappush(heap, (node.next.val, idx, node.next))

    pass  # replace with: return dummy.next

In [ ]:
# Uncomment and run when solution is ready
# test_harness(merge_k_lists)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (collect + sort) | O(N log N) | O(N) |
| Merge two at a time naively | O(kN) | O(1) |
| **Min-heap (optimal)** | **O(N log k)** | **O(k)** |
| Divide & conquer merge | O(N log k) | O(log k) stack |

Where N = total nodes, k = number of lists.  
The heap always holds at most k elements.

## Real World Connection

At Citi, transaction logs arrive from multiple trading desks each
pre-sorted by timestamp — merging them into one audit stream is
exactly this problem. AWS Kinesis and Kafka consumers often merge
ordered partitions the same way when building a unified event
timeline. In data engineering, external merge sort on large files
uses this k-way merge as its final pass to produce a single sorted
output. Search engines merge posting lists from different index
shards using the identical min-heap technique.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra